# Mini-TP 7 · Seguridad con SAIF — Data Lake (MinIO/S3)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Data Lake (MinIO/S3)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **exfiltración** de datos/modelos y acceso excesivo (demasiados permisos).  
**Controles:** acceso mínimo por bucket/prefijo · versionado + *checksum* (integridad) · URLs prefirmadas con expiración.  
**Elementos SAIF:** bases sólidas · coherencia de controles.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**; solo necesita **MinIO** levantado en Docker.


### Dependencias y MinIO
```bash
uv add boto3 scikit-learn joblib
```
MinIO tiene que estar corriendo en `http://localhost:9000` (ver README de la clase 6). Si el contenedor corre sin permisos de escritura sobre `/data`, agregá `--user root` al `docker run`.

In [1]:
import os, io, hashlib, json, joblib, numpy as np
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
ENDPOINT = "http://localhost:9000"; BUCKET = "datalake-seg"

import boto3
from botocore.client import Config
from botocore.exceptions import ClientError
s3 = boto3.client("s3", endpoint_url=ENDPOINT,
                  config=Config(signature_version="s3v4", s3={"addressing_style": "path"},
                                connect_timeout=5, retries={"max_attempts": 1}))
try:
    s3.create_bucket(Bucket=BUCKET)
except ClientError as e:
    if e.response["Error"]["Code"] not in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        raise
except Exception as exc:
    raise RuntimeError(f"MinIO no está disponible en {ENDPOINT}: {exc}") from exc

# un "modelo" de ejemplo para el control guiado
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
d = load_breast_cancer(); modelo = LogisticRegression(max_iter=5000).fit(d.data, d.target)
print("Bucket listo:", BUCKET)

Bucket listo: datalake-seg


## 0. La versión vulnerable
Un bucket con acceso amplio y un modelo cargado **sin verificar** deja dos riesgos abiertos:
- **Exfiltración / acceso excesivo:** cualquier credencial lee o escribe todo el bucket (modelos, datos crudos, auditoría).
- **Manipulación del artefacto:** si alguien reemplaza `modelo.pkl`, el servicio lo carga igual. Con `pickle`/`joblib` eso es además **ejecución remota de código**, porque deserializar ejecuta lo que el archivo diga.

Las secciones siguientes cierran estos riesgos con tres controles: integridad, acceso mínimo y trazabilidad.

## 1. Integridad — subir el modelo con checksum y verificarlo al cargar

In [2]:
buf = io.BytesIO(); joblib.dump(modelo, buf); blob = buf.getvalue()
sha = hashlib.sha256(blob).hexdigest()
s3.put_object(Bucket=BUCKET, Key="models/v1/modelo.pkl", Body=blob)
s3.put_object(Bucket=BUCKET, Key="models/v1/modelo.sha256", Body=sha.encode())

def cargar_verificando():
    blob = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.pkl")["Body"].read()
    esperado = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.sha256")["Body"].read().decode()
    if hashlib.sha256(blob).hexdigest() != esperado:
        raise ValueError("INTEGRIDAD ROTA: el artefacto fue manipulado")
    return joblib.load(io.BytesIO(blob))

m = cargar_verificando()
print("Modelo cargado y verificado por checksum. SHA:", sha[:20], "...")

Modelo cargado y verificado por checksum. SHA: d380af47ca26e08f1800 ...


In [3]:
# Prueba del control: si el artefacto cambia en el lake, la carga se rechaza ANTES de deserializarlo.
original = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.pkl")["Body"].read()
s3.put_object(Bucket=BUCKET, Key="models/v1/modelo.pkl", Body=original[:-1] + b"X")   # simula manipulación
try:
    cargar_verificando()
    print("ERROR: la manipulación NO fue detectada")
except ValueError as e:
    print("Detectado:", e)
finally:
    s3.put_object(Bucket=BUCKET, Key="models/v1/modelo.pkl", Body=original)             # se restaura
print("Tras restaurar, carga OK:", type(cargar_verificando()).__name__)

Detectado: INTEGRIDAD ROTA: el artefacto fue manipulado
Tras restaurar, carga OK: LogisticRegression


## 2. Control principal (GUIADO) — URL prefirmada con expiración (en vez de bucket público)

In [4]:
# En vez de exponer el bucket, se entrega un enlace temporal y acotado:
url = s3.generate_presigned_url("get_object",
        Params={"Bucket": BUCKET, "Key": "models/v1/modelo.pkl"},
        ExpiresIn=60)   # válido 60 segundos
print("URL prefirmada (expira en 60 s):")
print(url[:90], "...")

# Política de acceso MÍNIMO (ejemplo conceptual): solo lectura de models/ para un rol.
politica = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow", "Action": ["s3:GetObject"],
        "Resource": f"arn:aws:s3:::{BUCKET}/models/*"   # nada más que models/
    }]
}
print("\nPolítica de acceso mínimo (solo lectura de models/):")
print(json.dumps(politica, indent=2))

URL prefirmada (expira en 60 s):
http://localhost:9000/datalake-seg/models/v1/modelo.pkl?X-Amz-Algorithm=AWS4-HMAC-SHA256&X ...

Política de acceso mínimo (solo lectura de models/):
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::datalake-seg/models/*"
    }
  ]
}


In [5]:
# Prueba de la URL prefirmada: funciona, expira, y no sirve para otra clave.
import time, urllib.request, urllib.error

def estado(u):
    try:
        with urllib.request.urlopen(u, timeout=5) as r:
            return r.status
    except urllib.error.HTTPError as e:
        return e.code

corta = s3.generate_presigned_url("get_object",
        Params={"Bucket": BUCKET, "Key": "models/v1/modelo.pkl"}, ExpiresIn=2)
print("Recién generada     ->", estado(corta))                     # 200
time.sleep(3)
print("Después de expirar  ->", estado(corta))                     # 403

valida = s3.generate_presigned_url("get_object",
        Params={"Bucket": BUCKET, "Key": "models/v1/modelo.pkl"}, ExpiresIn=60)
otra = valida.replace("models/v1/modelo.pkl", "models/v1/modelo.sha256")
print("Cambiando la clave  ->", estado(otra))                      # 403: la firma no coincide

Recién generada     -> 200
Después de expirar  -> 403
Cambiando la clave  -> 403


## 3. Tu entrega — los `# TODO` resueltos

### TODO 1 · Mi modelo en el lake, con checksum y verificación al cargar
Se usa el modelo del **Mini-TP 6** (`LogisticRegression` sobre `digits`, guardado en `datalake-tp6/models/v1/`). Si ese bucket no existe, se reentrena el mismo modelo. Más abajo se sube con su SHA-256 pasando por el control de acceso (roles) y se verifica al cargarlo.

In [6]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

try:   # se reutiliza el artefacto del TP6 si está disponible
    blob_tp6 = s3.get_object(Bucket="datalake-tp6", Key="models/v1/logreg_digits.pkl")["Body"].read()
    mi_modelo = joblib.load(io.BytesIO(blob_tp6))
    origen = "cargado desde datalake-tp6/models/v1"
except Exception:
    from sklearn.linear_model import LogisticRegression as LR
    mi_modelo = LR(max_iter=500).fit(Xtr, ytr)
    origen = "reentrenado (datalake-tp6 no disponible)"
acc = float((mi_modelo.predict(Xte) == yte).mean())
print(f"Mi modelo ({origen}): {type(mi_modelo).__name__}, accuracy = {acc:.3f}")

Mi modelo (cargado desde datalake-tp6/models/v1): LogisticRegression, accuracy = 0.971


### TODO 3 · Zonas con permisos distintos (acceso mínimo)
Cada **rol** recibe solo lo que necesita (política *deny by default*). El bucket se organiza por zonas:

| Rol | Qué hace | Permisos |
|---|---|---|
| `ingestor` | carga datos nuevos | solo **escribir** en `raw/` |
| `trainer` | entrena y publica modelos | **escribir** en `curated/` y `models/` |
| `serving` | sirve predicciones | solo **leer** `curated/` y `models/` |
| `auditor` | revisa la trazabilidad | solo **leer** `audit/` |

> **Alcance:** esta política se aplica en la capa de aplicación (clase `LakeSeguro`). En producción las mismas sentencias se cargarían como **políticas IAM** de MinIO/S3 asociadas a usuarios o roles distintos, de modo que el almacenamiento las haga cumplir aunque falle la aplicación. Acá se usa una única credencial de servicio, así que esto demuestra el diseño de la política y no reemplaza al control del servidor.

### TODO 2 · Registro de acceso (audit log) — medida de gobernanza
Cada `get`/`put` —**permitido o denegado**— queda registrado con `(ts, rol, operación, key, resultado)` y, si hubo transferencia, el SHA-256 del contenido. El log se escribe en la zona `audit/` del lake como **JSON Lines**, un objeto nuevo por vuelco (nunca se sobreescribe uno anterior).

In [7]:
import fnmatch, datetime as dt

ACCIONES = {"get": "s3:GetObject", "put": "s3:PutObject"}

def stmt(action, *prefijos):
    return {"Effect": "Allow", "Action": [action],
            "Resource": [f"arn:aws:s3:::{BUCKET}/{p}*" for p in prefijos]}

POLITICAS = {
    "ingestor": [stmt("s3:PutObject", "raw/")],
    "trainer":  [stmt("s3:PutObject", "curated/", "models/")],
    "serving":  [stmt("s3:GetObject", "curated/", "models/")],
    "auditor":  [stmt("s3:GetObject", "audit/")],
}
print("Política del rol 'serving':")
print(json.dumps(POLITICAS["serving"], indent=2))


class AccesoDenegado(PermissionError):
    pass


class LakeSeguro:
    """Cliente S3 con acceso mínimo por rol y registro de auditoría de cada operación."""

    def __init__(self, cliente, bucket, rol, politicas=POLITICAS):
        self.s3, self.bucket, self.rol = cliente, bucket, rol
        self.politica = politicas[rol]
        self.eventos = []

    def _permitido(self, op, key):
        arn = f"arn:aws:s3:::{self.bucket}/{key}"
        return any(
            s["Effect"] == "Allow" and ACCIONES[op] in s["Action"]
            and any(fnmatch.fnmatch(arn, r) for r in s["Resource"])
            for s in self.politica)                      # sin sentencia que lo permita -> denegado

    def _registrar(self, op, key, resultado, contenido=None):
        ev = {"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="milliseconds"),
              "rol": self.rol, "op": op, "key": key, "resultado": resultado}
        if contenido is not None:
            ev["sha256"] = hashlib.sha256(contenido).hexdigest()
        self.eventos.append(ev)

    def put(self, key, body):
        if not self._permitido("put", key):
            self._registrar("put", key, "DENY")
            raise AccesoDenegado(f"el rol '{self.rol}' no puede escribir en {key}")
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=body)
        self._registrar("put", key, "ALLOW", body)

    def get(self, key):
        if not self._permitido("get", key):
            self._registrar("get", key, "DENY")
            raise AccesoDenegado(f"el rol '{self.rol}' no puede leer {key}")
        body = self.s3.get_object(Bucket=self.bucket, Key=key)["Body"].read()
        self._registrar("get", key, "ALLOW", body)
        return body

    def volcar_auditoria(self):
        """Persiste los eventos pendientes en audit/ (objeto nuevo por vuelco, append-only)."""
        if not self.eventos:
            return None
        ahora = dt.datetime.now(dt.timezone.utc)
        key = f"audit/{ahora:%Y-%m-%d}/{ahora:%H%M%S%f}_{self.rol}.jsonl"
        self.s3.put_object(Bucket=self.bucket, Key=key,
                           Body="\n".join(json.dumps(e) for e in self.eventos).encode())
        self.eventos = []
        return key


trainer = LakeSeguro(s3, BUCKET, "trainer")
serving = LakeSeguro(s3, BUCKET, "serving")
ingestor = LakeSeguro(s3, BUCKET, "ingestor")
auditor = LakeSeguro(s3, BUCKET, "auditor")
print("\nRoles listos:", list(POLITICAS))

Política del rol 'serving':
[
  {
    "Effect": "Allow",
    "Action": [
      "s3:GetObject"
    ],
    "Resource": [
      "arn:aws:s3:::datalake-seg/curated/*",
      "arn:aws:s3:::datalake-seg/models/*"
    ]
  }
]

Roles listos: ['ingestor', 'trainer', 'serving', 'auditor']


#### Publicar mi modelo con checksum y cargarlo verificando (TODO 1, pasando por los roles)

In [8]:
buf = io.BytesIO(); joblib.dump(mi_modelo, buf); blob = buf.getvalue()
sha = hashlib.sha256(blob).hexdigest()

# el trainer publica el artefacto y su checksum
trainer.put("models/v1/logreg_digits.pkl", blob)
trainer.put("models/v1/logreg_digits.sha256", sha.encode())

def cargar_modelo_verificado(lake, key="models/v1/logreg_digits.pkl"):
    """Descarga el artefacto, verifica su SHA-256 y recién entonces lo deserializa."""
    contenido = lake.get(key)
    esperado = lake.get(key.replace(".pkl", ".sha256")).decode().strip()
    if hashlib.sha256(contenido).hexdigest() != esperado:
        raise ValueError("INTEGRIDAD ROTA: el artefacto fue manipulado, no se carga")
    return joblib.load(io.BytesIO(contenido))

# el serving (solo lectura) carga y sirve
m_serv = cargar_modelo_verificado(serving)
acc_serv = float((m_serv.predict(Xte) == yte).mean())
print(f"SHA-256 publicado: {sha[:24]}...")
print(f"Modelo servido por el rol 'serving': accuracy = {acc_serv:.3f} (original: {acc:.3f})")
assert acc_serv == acc

# manipulación del artefacto en el lake (con credenciales de administración) -> el serving la rechaza
s3.put_object(Bucket=BUCKET, Key="models/v1/logreg_digits.pkl", Body=blob[:-1] + b"X")
try:
    cargar_modelo_verificado(serving); print("ERROR: no se detectó la manipulación")
except ValueError as e:
    print("Detectado:", e)
s3.put_object(Bucket=BUCKET, Key="models/v1/logreg_digits.pkl", Body=blob)   # se restaura
print("Restaurado, carga OK:", type(cargar_modelo_verificado(serving)).__name__)

SHA-256 publicado: 9fd93c1badb9e07ee801c3bc...
Modelo servido por el rol 'serving': accuracy = 0.971 (original: 0.971)
Detectado: INTEGRIDAD ROTA: el artefacto fue manipulado, no se carga
Restaurado, carga OK: LogisticRegression


#### Matriz de permisos: lo permitido funciona y lo no permitido se deniega (y queda registrado)

In [9]:
import pandas as pd

pruebas = [  # (cliente, op, key, ¿debería permitirse?)
    (ingestor, "put", "raw/digits/lote1.csv",        True),
    (ingestor, "put", "models/v1/logreg_digits.pkl", False),   # el ingestor no toca modelos
    (ingestor, "get", "raw/digits/lote1.csv",        False),   # ni siquiera puede leer lo que escribe
    (trainer,  "put", "curated/digits/train.parquet", True),
    (trainer,  "get", "raw/digits/lote1.csv",        False),
    (serving,  "get", "models/v1/logreg_digits.pkl", True),
    (serving,  "put", "models/v2/logreg_digits.pkl", False),   # el serving no puede publicar ni pisar modelos
    (serving,  "get", "raw/digits/lote1.csv",        False),   # exfiltración de datos crudos
    (serving,  "get", "audit/x.jsonl",               False),   # no puede leer la auditoría
]
filas = []
for lake, op, key, esperado in pruebas:
    try:
        lake.put(key, b"datos de prueba") if op == "put" else lake.get(key)
        obtenido = True
    except AccesoDenegado:
        obtenido = False
    filas.append({"rol": lake.rol, "op": op, "key": key,
                  "esperado": "ALLOW" if esperado else "DENY",
                  "obtenido": "ALLOW" if obtenido else "DENY", "ok": obtenido == esperado})
tabla = pd.DataFrame(filas)
print(tabla.to_string(index=False))
assert tabla["ok"].all(), "alguna regla de acceso no se comporta como se esperaba"
print("\nTodas las reglas se cumplen.")

     rol  op                          key esperado obtenido   ok
ingestor put         raw/digits/lote1.csv    ALLOW    ALLOW True
ingestor put  models/v1/logreg_digits.pkl     DENY     DENY True
ingestor get         raw/digits/lote1.csv     DENY     DENY True
 trainer put curated/digits/train.parquet    ALLOW    ALLOW True
 trainer get         raw/digits/lote1.csv     DENY     DENY True
 serving get  models/v1/logreg_digits.pkl    ALLOW    ALLOW True
 serving put  models/v2/logreg_digits.pkl     DENY     DENY True
 serving get         raw/digits/lote1.csv     DENY     DENY True
 serving get                audit/x.jsonl     DENY     DENY True

Todas las reglas se cumplen.


#### Auditoría: el log queda en el lake y el `auditor` puede leerlo

In [10]:
# cada rol vuelca sus eventos a audit/
claves = [k for k in (l.volcar_auditoria() for l in (ingestor, trainer, serving)) if k]
print("Objetos de auditoría escritos:")
for k in claves:
    print("  ", k)

# el auditor (solo lectura sobre audit/) reconstruye la historia
eventos = []
for k in claves:
    eventos += [json.loads(l) for l in auditor.get(k).decode().splitlines()]
log = pd.DataFrame(eventos).sort_values("ts")
log["sha256"] = log["sha256"].str[:10].fillna("-")
print(f"\n{len(log)} eventos | permitidos: {(log.resultado == 'ALLOW').sum()} | denegados: {(log.resultado == 'DENY').sum()}")
print(log[["ts", "rol", "op", "key", "resultado", "sha256"]].to_string(index=False))

print("\nIntentos denegados (lo que un SOC revisaría primero):")
print(log[log.resultado == "DENY"][["ts", "rol", "op", "key"]].to_string(index=False))

Objetos de auditoría escritos:
   audit/2026-10-13/004924792717_ingestor.jsonl
   audit/2026-10-13/004924817738_trainer.jsonl
   audit/2026-10-13/004924836130_serving.jsonl

17 eventos | permitidos: 11 | denegados: 6
                           ts      rol  op                            key resultado     sha256
2026-10-13T00:49:24.558+00:00  trainer put    models/v1/logreg_digits.pkl     ALLOW 9fd93c1bad
2026-10-13T00:49:24.584+00:00  trainer put models/v1/logreg_digits.sha256     ALLOW 401da2c9b4
2026-10-13T00:49:24.590+00:00  serving get    models/v1/logreg_digits.pkl     ALLOW 9fd93c1bad
2026-10-13T00:49:24.595+00:00  serving get models/v1/logreg_digits.sha256     ALLOW 401da2c9b4
2026-10-13T00:49:24.632+00:00  serving get    models/v1/logreg_digits.pkl     ALLOW 5acdaae1ec
2026-10-13T00:49:24.640+00:00  serving get models/v1/logreg_digits.sha256     ALLOW 401da2c9b4
2026-10-13T00:49:24.674+00:00  serving get    models/v1/logreg_digits.pkl     ALLOW 9fd93c1bad
2026-10-13T00:49:24.679

## 4. Reflexión

**1. ¿Por qué una URL prefirmada con expiración es más segura que un bucket público?**
Un bucket público deja el contenido abierto a cualquiera, para siempre y sin saber quién accedió. La URL prefirmada es una **credencial temporal, acotada a una sola clave y una sola operación**: en la prueba, a los 3 segundos de generarla con `ExpiresIn=2` devolvió 403, y al cambiarle la clave por otra también (la firma ya no coincide). Si el enlace se filtra, el daño queda limitado a un objeto y a una ventana corta. Además, cada emisión pasa por código propio, donde se puede autenticar a quien la pide, aplicar la política de roles y registrarla en el audit log. Su límite: quien tenga el enlace durante la ventana puede usarlo, por eso conviene que sea de vida corta.

**2. ¿Qué ataque detecta el checksum del modelo?**
Detecta la **manipulación o sustitución del artefacto** (*model tampering* / envenenamiento de la cadena de suministro): alguien con acceso de escritura al lake reemplaza el `.pkl` por una versión alterada, o el archivo se corrompe. En la prueba, cambiar un solo byte hizo que `cargar_modelo_verificado` lo rechazara **antes de deserializarlo**, lo cual importa porque cargar un `pickle`/`joblib` ejecuta código arbitrario. Limitaciones que hay que reconocer: (a) el checksum vive en el mismo bucket, así que un atacante con permiso de escritura sobre `models/` puede reemplazar ambos; se refuerza guardando el hash en otro sistema, **firmándolo** (HMAC o firma digital con una clave que el lake no tiene) y negando la escritura en `models/` a todo rol salvo `trainer`, como hace la política de acceso mínimo; (b) el checksum prueba que el archivo no cambió, no que el modelo sea bueno ni que su origen fuera legítimo.

**3. ¿Qué elemento de SAIF cubre tu control principal?**
Los controles se apoyan en distintos elementos de SAIF:
- **Ampliar las bases sólidas de seguridad al ecosistema de IA** → *acceso mínimo por rol y zona*, *URLs prefirmadas* y *checksum*. Es el control principal: aplica principios clásicos (mínimo privilegio, credenciales de vida corta, integridad) a los artefactos y datos de ML. En la matriz de permisos, el `serving` no puede leer `raw/` (exfiltración) ni escribir en `models/` (sustitución de modelo).
- **Ampliar la detección y la respuesta** → *audit log*. Registrar también los accesos **denegados** permite detectar sondeos y credenciales mal usadas.
- **Armonizar los controles a nivel de plataforma (coherencia)** → la misma política de roles por zona (`raw`/`curated`/`models`/`audit`) se puede reutilizar en el resto de las capas (serving, streaming, federado) para que los controles sean consistentes.

**Amenaza principal que mitigo:** exfiltración de datos y modelos por acceso excesivo, y sustitución del modelo servido.

**Limitaciones de esta entrega.**
- La política se aplica en la aplicación con una única credencial de servicio; un control real requiere políticas IAM del lado de MinIO/S3 con usuarios separados.
- El audit log solo cubre los accesos que pasan por `LakeSeguro`; las manipulaciones hechas directamente con credenciales de administración (como la simulación del ataque en este notebook) no quedan registradas. En producción se complementa con los **logs de auditoría del propio servidor**, notificaciones de bucket y versionado/*object lock* del bucket.